In [5]:
import random
import math
import matplotlib.pyplot as plt


# ---------- Lectura / generación del grafo ----------

def validar_tamano(n):
    if not (1 <= n < 21):
        raise ValueError(f"N debe ser menor que 21 (recibido: {n})")


def validar_tamano(n):
    if not (1 <= n < 21):
        print(f"Error: N debe ser menor que 21 (recibido: {n})")
        return False
    return True


def leer_matriz_archivo(ruta):
    with open(ruta, 'r') as f:
        lineas = [l.split() for l in f if l.strip()]
    matriz = [[int(x) for x in fila] for fila in lineas]
    n = len(matriz)
    if not validar_tamano(n):
        return None
    for fila in matriz:
        if len(fila) != n:
            print("Error: la matriz no es NxN")
            return None
    for i in range(n):
        for j in range(n):
            if matriz[i][j] != matriz[j][i]:
                print("Error: la matriz no es simetrica")
                return None
    return matriz


def generar_matriz_aleatoria(n, prob_arista=0.3, semilla=None):
    if not validar_tamano(n):
        return None
    if semilla is not None:
        random.seed(semilla)
    m = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            if random.random() < prob_arista:
                m[i][j] = m[j][i] = 1
    return m


def guardar_matriz_archivo(matriz, ruta):
    with open(ruta, 'w') as f:
        for fila in matriz:
            f.write(' '.join(str(x) for x in fila) + '\n')


def vecinos(matriz, nodo):
    return [j for j, v in enumerate(matriz[nodo]) if v == 1]


# ---------- Dynamic Ordering (MRV, desempate por grado) ----------

def seleccionar_variable(matriz, dominios, asignacion):
    no_asignados = [v for v in dominios if v not in asignacion]
    return min(
        no_asignados,
        key=lambda v: (len(dominios[v]), -len(vecinos(matriz, v)))
    )


# ---------- Forward Checking ----------

def forward_checking(matriz, dominios, nodo, color, asignacion):
    eliminados = {}
    for vecino in vecinos(matriz, nodo):
        if vecino not in asignacion and color in dominios[vecino]:
            dominios[vecino].remove(color)
            eliminados.setdefault(vecino, []).append(color)
            if len(dominios[vecino]) == 0:
                return False, eliminados
    return True, eliminados


def restaurar(dominios, eliminados):
    for var, colores in eliminados.items():
        dominios[var].extend(colores)


# ---------- BT-FC-DO ----------

def bt_fc_do(matriz, k):
    n = len(matriz)
    dominios = {i: list(range(1, k + 1)) for i in range(n)}
    asignacion = {}

    def backtrack():
        if len(asignacion) == n:
            return dict(asignacion)

        var = seleccionar_variable(matriz, dominios, asignacion)

        for color in list(dominios[var]):
            asignacion[var] = color
            ok, eliminados = forward_checking(matriz, dominios, var, color, asignacion)

            if ok:
                resultado = backtrack()
                if resultado is not None:
                    return resultado

            restaurar(dominios, eliminados)
            del asignacion[var]

        return None

    return backtrack()


def validar_coloreo(matriz, asignacion):
    n = len(matriz)
    for i in range(n):
        for j in range(n):
            if matriz[i][j] == 1 and asignacion.get(i) == asignacion.get(j):
                return False
    return True


def numero_cromatico(matriz):
    n = len(matriz)
    for k in range(1, n + 1):
        resultado = bt_fc_do(matriz, k)
        if resultado is not None:
            return k, resultado
    return n, None


# ---------- Visualizacion ----------

PALETA = [
    "#e74c3c", "#3498db", "#2ecc71", "#f1c40f", "#9b59b6",
    "#e67e22", "#1abc9c", "#34495e", "#95a5a6", "#d35400",
]


def graficar_coloreo(matriz, asignacion, titulo="Coloreado de grafo", ruta="coloreo.png"):
    n = len(matriz)

    pos = {}
    for i in range(n):
        angulo = 2 * math.pi * i / n
        pos[i] = (math.cos(angulo), math.sin(angulo))

    fig, ax = plt.subplots(figsize=(7, 7))

    for i in range(n):
        for j in range(i + 1, n):
            if matriz[i][j] == 1:
                x1, y1 = pos[i]
                x2, y2 = pos[j]
                ax.plot([x1, x2], [y1, y2], color="#555555", zorder=1, linewidth=1.2)

    for i in range(n):
        x, y = pos[i]
        color = PALETA[(asignacion[i] - 1) % len(PALETA)]
        ax.scatter(x, y, s=1200, color=color, zorder=2, edgecolors="white", linewidths=1.5)
        ax.text(x, y, str(i), ha="center", va="center", color="white", fontweight="bold", fontsize=12, zorder=3)

    ax.set_title(f"{titulo} (k={max(asignacion.values())} colores)")
    ax.set_aspect("equal")
    ax.axis("off")

    plt.savefig(ruta, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Figura guardada en: {ruta}")


# ---------- Demo ----------

if __name__ == "__main__":
    n = 6
    matriz = generar_matriz_aleatoria(n, prob_arista=0.4, semilla=42)

    if matriz is not None:
        print("Matriz de adyacencia:")
        for fila in matriz:
            print(fila)

        k = 3
        resultado = bt_fc_do(matriz, k)
        print(f"\nSolucion con k={k} colores:")
        print(resultado if resultado else "No existe solucion con ese k")

        k_min, coloreo = numero_cromatico(matriz)
        print(f"\nNumero cromatico encontrado: {k_min}")
        print("Coloreo:", coloreo)
        print("Coloreo valido:", validar_coloreo(matriz, coloreo))

        graficar_coloreo(matriz, coloreo, titulo="Coloreado optimo", ruta="coloreo.png")

Matriz de adyacencia:
[0, 0, 1, 1, 1, 0]
[0, 0, 0, 0, 1, 0]
[1, 0, 0, 1, 1, 0]
[1, 0, 1, 0, 1, 1]
[1, 1, 1, 1, 0, 0]
[0, 0, 0, 1, 0, 0]

Solucion con k=3 colores:
No existe solucion con ese k

Numero cromatico encontrado: 4
Coloreo: {3: 1, 4: 2, 0: 3, 2: 4, 1: 1, 5: 2}
Coloreo valido: True
Figura guardada en: coloreo.png
